# Research workflow — public data and frozen-source evidence

Owner: repository founder. Current workflow: 2026-09-22.

Choose a workflow in the next cell, restart the kernel, and **Run All**:

- **`p1-evidence` (default):** reopen the explicitly pinned Binance sample, verify
  native capture evidence, inspect strict source admission, and retain a QR handoff.
  This route makes no network call. It addresses the [frozen P1 requirements](../docs/research/p1-baseline.md#acceptance-and-research-discipline).
- **`book-capture`:** obtain one new bounded public Bitfinex R0 recording and inspect
  visible-size imbalance. This is a separate research question and cannot close P1.

Every run creates an exclusive ignored study. Missing or corrupt selected inputs
stop execution without switching sources. The evidence handoff can be completed
while **MR 8 remains open**: source facts and strict platform admission must pass
before an actual-source economic acceptance run is possible.

[Offline research](research-foundations.ipynb) · [Run and retain](notebook-operations.md#frozen-source-evidence)

## 1. Select the research question

The P1 sample and both economic configurations stay frozen. This is an unfitted
engineering sample; no parameter search, training split or holdout is introduced.
The book route retains its independent quantity-imbalance question.

In [ ]:
# standard library imports
import os

workflow = os.environ.get("ALPHA_QR_WORKFLOW", "p1-evidence")
if workflow not in {"p1-evidence", "book-capture"}:
    raise ValueError("Select p1-evidence or book-capture; no automatic fallback")
if workflow == "p1-evidence":
    question = {
        "question": (
            "Which frozen P1 source requirements have evidence, and what blocks acceptance?"
        ),
        "sample": "Binance BTCUSDT spot, 2024-06-01 [00:00, 01:00) UTC",
        "comparator": "unchanged frozen no-trade control",
        "fitting": "inapplicable; unfitted engineering sample",
        "holdout": "not designated or opened",
        "stop_if": "selected files are absent, corrupt or outside the frozen specification",
    }
else:
    question = {
        "question": (
            "What visible-size imbalance is observable in this bounded received order window?"
        ),
        "sample": "one new public Bitfinex R0 recording; first 32 raw positions for analysis",
        "observable": "(visible buy quantity - visible sell quantity) / total visible quantity",
        "horizon": "individual received states; no forward label",
        "comparator": "zero imbalance; no performance claim",
        "stop_if": "capture or publication fails; absent usable states remain unavailable",
        "fitting": "not applicable",
        "holdout": "not opened",
    }
print("Selected workflow:", workflow)
question

### Prepare the installed candidate

Use the pinned kernel, restart, then run in order. The complete installed inventory
and bytes must match this checkout. Identity includes the notebook source and attempt.

In [ ]:
# standard library imports
import hashlib
import json
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from fractions import Fraction
from importlib.metadata import distributions
from pathlib import Path
from uuid import uuid4

# third party library imports
import pandas as pd
from IPython.display import display
from msgspec.structs import replace

# project library imports
import market_data
from applications.explore import inspect_eda
from configuration import CaptureConfig
from market_data.acquisition import AcquisitionManifest
from market_data.adapters.sources import SourceAdapters
from market_data.contracts import ArchiveCapture, CaptureRequest, Order, SourceRequest
from market_data.datasets import DatasetInput, DatasetPublication
from market_data.query import BookMarketRow, BookQuery
from market_data.storage import RawStore
from market_data.validation import QualityPolicy, QualityReport
from reference_data.eligibility import EligibilityAssertions
from research.experiments import ResultManifest
from research.exploration import (
    BookResearchInput,
    BookSelection,
    ResearchDataset,
    ResearchReader,
    ResearchSelection,
)
from runtime import ImplementationIdentity, RuntimeFiles
from serialization import DomainCodec
from simulation import BacktestSpec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/live/bitfinex-btc-usd.json").is_file(), "Use the repository root/kernel"
assert platform.python_version() == (root / ".python-version").read_text().strip()
assert "site-packages" in Path(market_data.__file__).resolve().parts
codec = DomainCodec()
pd.set_option("display.max_colwidth", None)

In [ ]:
source_root = root / "src"
installed_inputs = dict(RuntimeFiles.inputs())
source_files = {
    path.relative_to(source_root).as_posix(): path
    for package in RuntimeFiles.roots
    for path in (source_root / package).rglob("*")
    if path.is_file() and (path.suffix in {".py", ".json"} or path.name == "py.typed")
}
assert source_files.keys() == installed_inputs.keys(), "Installed runtime inventory differs"
for name, source in source_files.items():
    assert source.read_bytes() == installed_inputs[name].read_bytes(), (
        f"Stale installed file: {name}"
    )
notebook_path = Path(
    os.environ.get(
        "ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations-live.ipynb"
    )
)
notebook = json.loads(notebook_path.read_text())
notebook_source = [(cell["cell_type"], "".join(cell["source"])) for cell in notebook["cells"]]
identity = {
    "commit": subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip(),
    "working_tree": "changed"
    if subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
    else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "tooling": {
        item.metadata["Name"]: item.version
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel", "pandas"}
    },
    "kernel": sys.executable,
    "package": market_data.__file__,
    "runtime_files": len(installed_inputs),
    "implementation_sha256": ImplementationIdentity.sha256(),
    "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "notebook_source_sha256": hashlib.sha256(
        json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
    ).hexdigest(),
    "attempt": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
}
display(pd.DataFrame(identity.items(), columns=["Identity", "Value"]))

## 2. Pin the frozen source and prepare a retained evidence handoff

**P1 route only.** Defaults name the previously retained acquisition and native witness
explicitly; they never search for the latest run. Set `ALPHA_P1_MANIFEST`,
`ALPHA_P1_RAW` and `ALPHA_P1_CAPTURE` to move those same inputs elsewhere.
An empty `ALPHA_P1_CAPTURE` deliberately runs an archive-only investigation.
A nonempty missing path is an error. On a clean checkout, first follow the
[saved archive acquisition command](../src/market_data/README.md#binance-public-daily-archive-acquisition).

Optional `ALPHA_P1_ELIGIBILITY` names a source-backed `EligibilityAssertions` file;
without it, eligibility remains unknown. The notebook never authors an inclusion
assertion from observed trades or current metadata. Optional `ALPHA_P1_DOCUMENTS`
names a completed copy of the generated `review-template.json`. Attach retained
provider records and your interpretation there; attachments remain **unreviewed
candidate evidence**, not an admission switch. `ALPHA_P1_REOPEN` names one previous
handoff directory for deterministic evidence reproduction in a fresh kernel.

In [ ]:
if workflow == "p1-evidence":
    p1_manifest_path = Path(
        os.environ.get(
            "ALPHA_P1_MANIFEST",
            root
            / "data/binance-archives/runs"
            / "acb948277201cc169d32c2bb874fb40ce9e9c33c893e56a992faa47853147f80"
            / "manifest.json",
        )
    ).resolve()
    p1_raw_path = Path(os.environ.get("ALPHA_P1_RAW", root / "data/binance-archives/raw")).resolve()
    p1_capture_path = os.environ.get(
        "ALPHA_P1_CAPTURE",
        str(root / "logs/frozen-baseline/2026-09-22/source-evidence/archive-capture.json"),
    )
    p1_eligibility_path = os.environ.get("ALPHA_P1_ELIGIBILITY", "")
    p1_documents_path = os.environ.get("ALPHA_P1_DOCUMENTS", "")
    p1_reopen_path = os.environ.get("ALPHA_P1_REOPEN", "")
    p1_max_bytes = 134_217_728
    p1_study = root / "data/notebooks/p1-evidence" / str(uuid4())
    p1_study.mkdir(parents=True, exist_ok=False)
    for name in ("raw", "snapshots", "documents"):
        (p1_study / name).mkdir()
    ResultManifest.export(
        p1_study / "attempt.json",
        codec.encode_json(
            {
                "identity": identity,
                "question": question,
                "started_at": datetime.now(UTC),
                "manifest": str(p1_manifest_path),
                "raw": str(p1_raw_path),
                "capture": p1_capture_path,
                "eligibility": p1_eligibility_path,
                "documents": p1_documents_path,
                "reopen": p1_reopen_path,
            }
        ),
    )
    print("P1 retained attempt:", p1_study)
    print("Archive acquisition:", p1_manifest_path)
    print("Native witness:", p1_capture_path or "explicitly absent")

In [ ]:
class SourceEvidenceFiles:
    """Retain bounded notebook evidence without overwriting earlier cell results."""

    def __init__(self, directory: Path) -> None:
        """Bind one exclusive notebook attempt and its semantic artifact inventory."""
        self.directory = directory
        self.digests: dict[str, str] = {}

    @staticmethod
    def read(path: Path, limit: int = 134_217_728) -> bytes:
        """Read a selected regular file with an explicit byte bound."""
        if not path.is_file():
            raise FileNotFoundError(f"Selected evidence file is missing: {path}")
        with path.open("rb") as handle:
            payload = handle.read(limit + 1)
        if len(payload) > limit:
            raise ValueError(f"Evidence exceeds {limit} bytes: {path}")
        return payload

    def retain(self, name: str, payload: bytes) -> None:
        """Save one artifact exclusively; rerunning requires a new setup attempt."""
        destination = self.directory / name
        if destination.exists():
            raise FileExistsError(f"Restart from setup to preserve this attempt: {destination}")
        ResultManifest.export(destination, payload)
        self.digests[name] = hashlib.sha256(payload).hexdigest()

    def quality(self, acquisition: str, report: QualityReport) -> None:
        """Retain the actual publisher report before its admission gate returns."""
        self.retain(
            "quality.json", codec.encode_json({"acquisition": acquisition, "report": report})
        )


if workflow == "p1-evidence":
    p1_files = SourceEvidenceFiles(p1_study)
    p1_expected_configs = {
        "p1-btcusdt-return-v1": "d1b9b17e5ff667c4cff86d197719d6a78c7b8d166d4755580b1ca656fae906b8",
        "p1-btcusdt-no-trade-v1": (
            "34cea2f4f03e9294a2a001c629bb0ca03edd0a8c259e67a9ca5b2d3599a20621"
        ),
    }
    p1_specs = {}
    for name, digest in p1_expected_configs.items():
        payload = p1_files.read(root / "configs/backtests" / f"{name}.json")
        if hashlib.sha256(payload).hexdigest() != digest:
            raise ValueError(f"Frozen configuration changed: {name}; review a versioned amendment")
        p1_specs[name] = codec.decode_json(payload, BacktestSpec)
        p1_files.retain(f"{name}.json", payload)
    p1_spec = p1_specs["p1-btcusdt-return-v1"]
    p1_control = p1_specs["p1-btcusdt-no-trade-v1"]
    assert replace(p1_control, instance=p1_spec.instance, rule=p1_spec.rule) == p1_spec
    p1_files.retain(
        "implementation.json",
        codec.encode_json(
            {
                "implementation": identity["implementation_sha256"],
                "lock": identity["lock_sha256"],
                "notebook_source": identity["notebook_source_sha256"],
            }
        ),
    )

In [ ]:
if workflow == "p1-evidence":
    p1_decoded = codec.decode_json(p1_files.read(p1_manifest_path), AcquisitionManifest)
    p1_saved = AcquisitionManifest.read(
        p1_manifest_path, run=p1_decoded.run, store=RawStore(p1_raw_path)
    )
    (p1_request,) = SourceRequest.partition(p1_saved.run)
    if (
        p1_saved.pending is not None
        or len(p1_saved.completed) != 1
        or (
            p1_request.instrument.source,
            p1_saved.run.job.stream,
            p1_request.instrument.instrument_id,
        )
        != p1_spec.binding
        or (p1_saved.run.job.start, p1_saved.run.job.end) != (p1_spec.start, p1_spec.end)
    ):
        raise ValueError("Acquisition must be the complete frozen Binance BTCUSDT hour")
    p1_original = p1_saved.completed[0]
    if p1_original.sha256 != "c9c333f58fe33faaa0a34d526c61b13f5648b908c4ad341dd6556e15c5bcc46a":
        raise ValueError(
            "Archive differs from the frozen retained bytes; investigate the source version"
        )
    p1_raw = RawStore(p1_study / "raw")
    p1_raw.publish(
        p1_raw_path / p1_original.sha256, descriptor=p1_original, request=p1_request.binding
    )
    p1_files.retain("acquisition.json", codec.encode_json(p1_saved))
    p1_files.digests[f"raw/{p1_original.sha256}"] = p1_original.sha256
    p1_capture = None
    if p1_capture_path:
        payload = p1_files.read(Path(p1_capture_path).resolve())
        p1_capture = codec.decode_json(payload, ArchiveCapture)
        if (
            p1_capture.request_identity != p1_request.identity
            or p1_capture.archive_sha256 != p1_original.sha256
        ):
            raise ValueError("Native witness belongs to a different request or archive")
        p1_files.retain("archive-capture.json", payload)
    p1_eligibility = (
        codec.decode_json(p1_files.read(Path(p1_eligibility_path).resolve()), EligibilityAssertions)
        if p1_eligibility_path
        else EligibilityAssertions(catalog=p1_saved.run.catalog, assertions=())
    )
    if p1_eligibility.catalog != p1_saved.run.catalog:
        raise ValueError("Eligibility must reference this acquisition's exact catalog")
    p1_files.retain("eligibility.json", codec.encode_json(p1_eligibility))
    p1_publisher = DatasetPublication(SourceAdapters(), ImplementationIdentity.sha256())
    p1_count = sum(1 for _ in p1_publisher.occurrences(p1_saved, p1_raw, max_bytes=p1_max_bytes))
    assert p1_count == 27_480, "Frozen selected row count differs"
    p1_files.retain(
        "source-summary.json",
        codec.encode_json(
            {
                "archive_sha256": p1_original.sha256,
                "selected_rows": p1_count,
                "start": p1_spec.start,
                "end": p1_spec.end,
                "capture_identity": None if p1_capture is None else p1_capture.identity,
                "eligibility_assertions": len(p1_eligibility.assertions),
                "scope": (
                    "verified saved bytes and selected rows; historical "
                    "completeness remains unproven"
                ),
            }
        ),
    )
    display(
        pd.DataFrame(
            [
                {
                    "selected_trades": p1_count,
                    "native_witness": "supplied; checked by admission below"
                    if p1_capture
                    else "absent",
                    "eligibility_assertions": len(p1_eligibility.assertions),
                    "frozen_configuration": "unchanged",
                }
            ]
        )
    )

## 3. Attach the QR's source investigation

The three rows below are the concrete questions to resolve with provider records.
For each supporting file, record its provenance, the exact claim it supports and
its limitations. A current listing, zero observed gaps, or matching archive/capture
rows cannot establish the stronger facts on their own.

The generated `review-template.json` is editable outside this attempt. Its `documents`
list accepts objects with `requirement`, absolute `path`, `source_reference`, `claim`
and `limitations`. Allowed requirements are the three keys in the table. Supply
the edited copy through `ALPHA_P1_DOCUMENTS` and restart. Files are copied and hashed;
their content and applicability still require review and supported domain contracts.

In [ ]:
if workflow == "p1-evidence":
    p1_requirements = {
        "historical_eligibility": (
            "BTCUSDT trading eligibility/reference facts throughout the "
            "hour, with historical availability times"
        ),
        "coverage": (
            "Positive interval-specific recorder health and completeness "
            "evidence, including omissions/outages"
        ),
        "revisions": (
            "Applicable original versions and correction history, "
            "including publication/availability times"
        ),
    }
    p1_scope = {
        "experiment": p1_spec.instance,
        "archive_sha256": p1_original.sha256,
        "start": p1_spec.start.isoformat(),
        "end": p1_spec.end.isoformat(),
    }
    p1_review = {"scope": p1_scope, "documents": []}
    if p1_documents_path:
        p1_review = json.loads(p1_files.read(Path(p1_documents_path).resolve(), 1_048_576))
        if set(p1_review) != {"scope", "documents"} or p1_review["scope"] != p1_scope:
            raise ValueError("Review must bind the exact frozen sample; use review-template.json")
    if not isinstance(p1_review["documents"], list) or len(p1_review["documents"]) > 20:
        raise ValueError("Supply at most 20 explicit supporting documents")
    p1_documents = []
    for index, record in enumerate(p1_review["documents"]):
        if (
            not isinstance(record, dict)
            or set(record) != {"requirement", "path", "source_reference", "claim", "limitations"}
            or any(not isinstance(value, str) or not value.strip() for value in record.values())
            or record["requirement"] not in p1_requirements
            or not Path(record["path"]).is_absolute()
        ):
            raise ValueError(
                "Each document requires a known requirement, absolute path, "
                "provenance, claim and limitations"
            )
        payload = p1_files.read(Path(record["path"]), 8_388_608)
        name = f"documents/{index:02d}-{Path(record['path']).name}"
        p1_files.retain(name, payload)
        p1_documents.append(
            {**record, "retained": name, "sha256": hashlib.sha256(payload).hexdigest()}
        )
    p1_files.retain(
        "document-review.json", codec.encode_json({"scope": p1_scope, "documents": p1_documents})
    )
    ResultManifest.export(
        p1_study / "review-template.json", codec.encode_json({"scope": p1_scope, "documents": []})
    )
    display(
        pd.DataFrame(
            [
                {
                    "requirement": name,
                    "evidence_needed": need,
                    "attached_documents": sum(item["requirement"] == name for item in p1_documents),
                    "status": "candidate documents require review"
                    if any(item["requirement"] == name for item in p1_documents)
                    else "missing",
                }
                for name, need in p1_requirements.items()
            ]
        )
    )

## 4. Ask the installed source gate, and retain its actual answer

This invokes `DatasetPublication.publish` with **strict `use="causal"`**, the
original archive, optional native capture and explicit eligibility history. It
never switches to `witnessed` or `exploratory` to obtain a pass. The publisher
re-decodes capture messages and checks agreement with every selected economic
version. Empty correction links mean none have been supplied, not that a complete
vendor revision history is known.

The current strict contract cannot admit complete coverage/revision evidence, so
rejection is an expected **OPEN** outcome. Unexpected I/O, decoding or programming
errors still fail the notebook. No price/NAV result is created from refused inputs.

In [ ]:
if workflow == "p1-evidence":
    p1_files.retain(
        "admission-request.json", codec.encode_json({"use": "causal", "max_bytes": p1_max_bytes})
    )
    p1_snapshot = None
    try:
        p1_snapshot = p1_publisher.publish(
            (
                DatasetInput(
                    acquisition=p1_saved,
                    policy=QualityPolicy(
                        use="causal",
                        known_at=p1_spec.start,
                        max_gap=timedelta(seconds=p1_spec.max_age_seconds),
                    ),
                    eligibility=p1_eligibility,
                    capture=p1_capture,
                ),
            ),
            p1_raw,
            p1_study / "snapshots",
            max_bytes=p1_max_bytes,
            report=p1_files.quality,
        )
    except ValueError:
        if not (p1_study / "quality.json").is_file():
            raise
        p1_quality = json.loads((p1_study / "quality.json").read_bytes())["report"]
        if p1_quality["scope"] != "rejected" or tuple((p1_study / "snapshots").iterdir()):
            raise
    p1_quality = json.loads((p1_study / "quality.json").read_bytes())["report"]
    p1_files.retain(
        "admission-outcome.json",
        codec.encode_json(
            {
                "snapshot": p1_snapshot,
                "scope": p1_quality["scope"],
                "reasons": p1_quality["reasons"],
            }
        ),
    )
    if p1_quality["invalid_count"] or any(
        reason.startswith(("capture evidence:", "revision admission:", "input "))
        for reason in p1_quality["reasons"]
    ):
        raise ValueError("Invalid source evidence; inspect retained quality.json before continuing")
    display(pd.DataFrame({"strict_admission_reason": p1_quality["reasons"]}))
    print(
        "Strict publication:",
        "REFUSED — baseline acceptance OPEN" if p1_snapshot is None else p1_snapshot,
    )

## 5. Reproduce the evidence handoff and assign the remaining work

Run this route in a fresh kernel with `ALPHA_P1_REOPEN` set to the first study to
verify all its retained artifact bytes and compare the new semantic inventory.
Attempt timestamps and UUIDs are separate from deterministic source/configuration/
quality artifacts. A match proves evidence reproduction, **not economic acceptance**.

The QR supplies source records and reviews applicability. The developer implements
the concrete missing coverage/revision admission contract, then uses the existing
[unfitted engineering-sample workflow](../src/research/experiments/README.md#unfitted-engineering-samples)
with both unchanged P1 configurations. Closure additionally needs actual-source
costed reports, exact no-trade/cash/position/NAV reconciliation and fresh-process
economic repeat/resume. Negative net returns are acceptable under the frozen rule.

If required evidence is unavailable, record that finding and propose a founder
decision about source/window or acceptance scope. This notebook does not amend it.

In [ ]:
if workflow == "p1-evidence":
    p1_inventory = dict(sorted(p1_files.digests.items()))
    p1_reproduction = "not requested"
    if p1_reopen_path:
        previous = Path(p1_reopen_path).resolve()
        prior_inventory = json.loads(p1_files.read(previous / "semantic-inventory.json", 1_048_576))
        for name, digest in prior_inventory.items():
            selected = (previous / name).resolve()
            if not selected.is_relative_to(previous):
                raise ValueError("Prior inventory path escapes its retained study")
            if hashlib.sha256(p1_files.read(selected)).hexdigest() != digest:
                raise ValueError(f"Prior evidence bytes changed: {name}")
        if prior_inventory != p1_inventory:
            raise ValueError(
                "Evidence inputs/code/quality changed; this is a new "
                "investigation, not reproduction"
            )
        p1_reproduction = "verified exact evidence artifacts in this fresh execution"
    ResultManifest.export(p1_study / "semantic-inventory.json", codec.encode_json(p1_inventory))
    p1_handoff = {
        "scope": p1_scope,
        "status": "MR 8 OPEN",
        "strict_snapshot": p1_snapshot,
        "quality_reasons": p1_quality["reasons"],
        "evidence_reproduction": p1_reproduction,
        "qr_next": p1_requirements,
        "candidate_documents": p1_documents,
        "developer_next": (
            "Review supplied facts and implement concrete strict "
            "coverage/revision admission; retain current refusal until "
            "supported"
        ),
        "economic_next": (
            "After source qualification, run both unchanged "
            "configurations through configured engineering-sample "
            "reports; reconcile and repeat/resume in fresh processes"
        ),
        "acceptance": (
            "No empirical economics or profitability result is "
            "established by this evidence notebook"
        ),
    }
    ResultManifest.export(p1_study / "handoff.json", codec.encode_json(p1_handoff))
    p1_note = "# Frozen P1 source-evidence handoff\n\nStatus: **MR 8 OPEN**.\n\n"
    p1_note += (
        "See `handoff.json`, `quality.json`, `document-review.json` "
        "and `semantic-inventory.json`.\n"
    )
    p1_note += (
        "The unchanged economic configurations, acquisition, raw "
        "archive and supplied witness are retained here.\n\n"
    )
    p1_note += "## Source gate\n\n" + "".join(f"- {reason}\n" for reason in p1_quality["reasons"])
    p1_note += "\n## QR source requests\n\n" + "".join(
        f"- {name}: {need}.\n" for name, need in p1_requirements.items()
    )
    p1_note += "\n" + p1_handoff["developer_next"] + ".\n\n" + p1_handoff["economic_next"] + ".\n"
    ResultManifest.export(p1_study / "handoff.md", p1_note.encode())
    print("QR handoff:", p1_study / "handoff.md")
    print("Attach provider records using:", p1_study / "review-template.json")
    print("Evidence reproduction:", p1_reproduction)
    print("For a fresh-kernel repeat, set ALPHA_P1_REOPEN to", p1_study)
    print("MR 8 remains OPEN; no actual-source economic result is claimed.")

## Book capture route — acquire and pin a bounded sample

**Runs only with `ALPHA_QR_WORKFLOW=book-capture`.**

The next configuration cell creates an exclusive retained study directory. The capture
cell contacts the configured public feed for at most 20 seconds of receipt, bounded by
messages, bytes, queues, idle time and retries. Completion/fsync can extend elapsed time;
the notebook also applies an 80-second child deadline. A timeout kills/waits for its child
and retains the partial recording for explicit interrupted inspection. No keys are read.
Review the displayed limits before running the capture cell interactively.

In [ ]:
if workflow == "book-capture":
    config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
    if config.venue != "bitfinex" or config.channel != "R0" or config.duration_seconds > 20:
        raise ValueError("This notebook selects public Bitfinex R0 with a receive interval <=20s")
    study = root / "data/notebooks/research-live" / str(uuid4())
    study.mkdir(parents=True)
    (study / "snapshots").mkdir()
    (study / "raw").mkdir()
    config_path = study / "capture-config.json"
    config_path.write_bytes(codec.encode_json(config))
    (study / "attempt.json").write_bytes(
        codec.encode_json({"identity": identity, "question": question})
    )
    display(
        pd.DataFrame(
            [
                {
                    "venue": config.venue,
                    "symbol": config.symbol,
                    "channel": config.channel,
                    "base": config.base_currency,
                    "quote": config.quote_currency,
                    "duration_seconds": config.duration_seconds,
                    "messages_max": config.max_messages,
                    "journal_bytes_max": config.max_bytes,
                    "frame_bytes_max": config.max_frame_bytes,
                    "retries": config.retries,
                    "published_raw_positions_max": 32,
                }
            ]
        )
    )
    print("Retained attempt directory:", study)

In [ ]:
if workflow == "book-capture":
    collector = Path(sys.executable).parent / "alpha-collect"
    if not collector.is_file():
        raise FileNotFoundError(f"Installed collector is missing: {collector}")
    command = [
        str(collector),
        "capture",
        "--config",
        str(config_path),
        "--output",
        str(study / "captures"),
    ]
    # Claim this attempt before any source effect; rerunning this cell requires new setup.
    with (study / "command.json").open("xb") as command_file:
        command_file.write(codec.encode_json(command))
    # Files retain partial diagnostics even if the child times out; no raw frames are displayed.
    with (
        (study / "capture.stdout").open("x") as stdout,
        (study / "capture.stderr").open("x") as stderr,
    ):
        capture = subprocess.run(
            command, cwd=study, stdout=stdout, stderr=stderr, timeout=80, check=False
        )
    (study / "capture-exit.json").write_bytes(codec.encode_json({"exit_code": capture.returncode}))
    if capture.returncode != 0:
        raise RuntimeError(
            f"Capture exit {capture.returncode}; inspect retained logs/prefix under {study}"
        )
    summary = json.loads((study / "capture.stdout").read_text())
    capture_directory = Path(summary["capture_directory"])
    request = codec.decode_json((capture_directory / "request.json").read_bytes(), CaptureRequest)
    assert request.config == config and request.identity == summary["request_identity"]
    display(
        pd.DataFrame(
            [
                {
                    name: summary[name]
                    for name in (
                        "venue",
                        "symbol",
                        "messages",
                        "epochs",
                        "gaps",
                        "stop_reason",
                        "replay_parity",
                    )
                }
            ]
        )
    )

Publication and input selection are separate explicit operations after successful capture.
The shared publisher verifies saved originals and admits only the first 32 raw positions;
that prefix may contain controls and fewer than 32 market states. One immutable selection
names its snapshot, request, fields and receipt/cutoff window. No latest-input search occurs.

In [ ]:
if workflow == "book-capture":
    # Copy the verified original into this study's raw/ store and publish a bounded prefix.
    snapshot_id = DatasetPublication(
        SourceAdapters(), ImplementationIdentity.sha256()
    ).publish_books(
        (capture_directory,),
        RawStore(study / "raw"),
        study / "snapshots",
        max_records=32,
        max_bytes=config.max_bytes,
    )
    # Query boundaries are microseconds; floor the start and ceil the excluded end.
    # Original per-message integer nanoseconds remain untouched in the result.
    epoch = datetime(1970, 1, 1, tzinfo=UTC)
    start = epoch + timedelta(microseconds=summary["started_ns"] // 1000)
    end = epoch + timedelta(microseconds=summary["finished_ns"] // 1000 + 1)
    selection = ResearchSelection(
        inputs=(
            BookSelection(
                name="visible_depth",
                query=BookQuery(
                    snapshot_id=snapshot_id,
                    request_identity=request.identity,
                    instrument=config.instrument_id,
                    start=start,
                    end=end,
                    known_at=end,
                    columns=("orders",),
                    batch_size=32,
                    max_bytes=config.max_bytes,
                ),
            ),
        )
    )
    selection_path = study / "selection.json"
    selection_path.write_bytes(codec.encode_json(selection))
    print("Snapshot:", snapshot_id)
    print("Selection:", selection.identity)
    print("UTC receipt interval [start, end):", start, end)

## 3. Inspect quality, missingness and receipt time

Use the same installed EDA entry as the offline version. Inspect controls beside values.
Book states remain exploratory/degraded, and absent states do not mean an empty market.
The final capture state is unavailable after session end even when reconstruction matched.

In [ ]:
if workflow == "book-capture":

    class StudyReports:
        """Retain EDA status beside the original live study."""

        @staticmethod
        def retain(record: dict[str, object]) -> None:
            """Append one application status record.

            Args:
                record: Application-owned status and input identities.
            """
            with (study / "eda-attempts.jsonl").open("ab") as output:
                output.write(codec.encode_json(record) + b"\n")

    result = inspect_eda(
        selection_path, study / "snapshots", RawStore(study / "raw"), report=StudyReports.retain
    )
    dataset = result["dataset"]
    assert isinstance(dataset, ResearchDataset) and dataset.identity == selection.identity
    depth = dataset.inputs[0]
    assert isinstance(depth, BookResearchInput)
    market_rows = [
        row for batch in depth.batches for row in batch.rows if isinstance(row, BookMarketRow)
    ]
    controls = [
        row for batch in depth.batches for row in batch.rows if not isinstance(row, BookMarketRow)
    ]
    display(
        pd.DataFrame(
            [
                {
                    "market_states": len(market_rows),
                    "control_boundaries": len(controls),
                    "base_quantity_unit": config.base_currency,
                    "price_quote_unit": config.quote_currency,
                    "coverage_limits": sorted(
                        {reason for batch in depth.batches for reason in batch.reasons}
                    ),
                }
            ]
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "position": row.position,
                    "received_ns": row.received_ns,
                    "status": row.status,
                    "reason": row.reason,
                }
                for row in controls
            ]
        )
    )

Move the cutoff before this recording. No subsequently received state may enter that
information set. Then reopen the original selection and compare the complete result.

In [ ]:
if workflow == "book-capture":
    reader = ResearchReader(study / "snapshots", RawStore(study / "raw"), readers=SourceAdapters())
    early = ResearchSelection(
        inputs=(
            replace(
                selection.inputs[0],
                query=replace(
                    selection.inputs[0].query, known_at=start - timedelta(microseconds=1)
                ),
            ),
        )
    )
    early_dataset = reader.read(early)
    assert not any(
        isinstance(row, BookMarketRow)
        for item in early_dataset.inputs
        for batch in item.batches
        for row in batch.rows
    )
    assert reader.read(selection) == dataset
    print("No values before this recording; original pinned selection reproduced.")

## 4. Prototype a feature; keep its label separate

For each admitted received L3 state, sum exact base quantities on each side and retain
the dimensionless imbalance as an exact rational. Use only these message-local visible
orders; do not forward-fill controls or infer missing depth. Zero total is unavailable.
Market-derived ratios and state identities are kept in the ignored study directory.
A later return label needs separately selected prices, horizon and availability evidence.

In [ ]:
if workflow == "book-capture":
    prototype = []
    for row in market_rows:
        orders = row.values[0]
        assert all(isinstance(order, Order) for order in orders)
        # Fractions preserve exact quantities independently of Decimal precision.
        bids = sum((Fraction(order.size) for order in orders if order.side == "buy"), Fraction())
        asks = sum((Fraction(order.size) for order in orders if order.side == "sell"), Fraction())
        total = bids + asks
        imbalance = None if total == 0 else (bids - asks) / total
        prototype.append(
            {
                "position": row.position,
                "received_ns": row.received_ns,
                "state": row.state_identity,
                "quality": row.status,
                "imbalance_exact_ratio": None if imbalance is None else str(imbalance),
            }
        )
    (study / "prototype.json").write_bytes(codec.encode_json(prototype))
    # Market-derived values stay in ignored evidence; notebook outputs show counts.
    print("Computed exploratory states:", len(prototype))
    print(
        "Unavailable denominators:",
        sum(item["imbalance_exact_ratio"] is None for item in prototype),
    )
    print("Exact ratios and state identities retained:", study / "prototype.json")

## Book capture route — hand off to the supported research pipeline

The exploratory R0 imbalance is not a compatible trade input for the implemented
feature/model/economic APIs. Retain its selection, original bytes, EDA statuses and
prototype for offline book inspection. The separate offline synthetic trade study
demonstrates retained features, models, configured economics and recovery.

For frozen P1 source investigation, select `p1-evidence` at the top and restart.
The Bitfinex capture contributes no Binance historical source evidence. Preserve
all influencing attempts under the [research record contract](../docs/roadmap/research-program.md#small-records-and-artifacts).

In [ ]:
if workflow == "book-capture":
    handoff = {
        "selection": selection.identity,
        "snapshot": snapshot_id,
        "request": request.identity,
        "capture": str(capture_directory),
        "market_states": len(market_rows),
        "status": "exploratory; no economic acceptance",
    }
    (study / "handoff.json").write_bytes(codec.encode_json(handoff))
    print("Offline QR: set ALPHA_RESEARCH_DATASET_DIRECTORY to", study)
    print("Offline engineer: set ALPHA_BITFINEX_CAPTURE_DIRECTORY to", capture_directory)
    print("Retained all inputs and attempts; no automatic deletion or repeated capture.")